<a href="https://colab.research.google.com/github/Saryace/rusle-ren237/blob/main/rusle_ren237.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este será el proyecto base que usaremos para crear un modelo RUSLE usando Google Earth Engine

In [ ]:
import math
import os

import ee
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

**Pasos para activar su cuenta de Google con un proyecto GEE**

1. Crear cuenta de Google (si no tienen una)

2. Ir a https://earthengine.google.com/signup/ e iniciar sesión con su cuenta de Google. Completar el formulario de registro seleccionando 'Uso no comercial / académico'
3. Crear un proyecto en Google Cloud Console
Ir a https://console.cloud.google.com/projectcreate, ponerle un nombre al proyecto (ej. 'clase-erosion-nombre') y anotar ese Project ID
4. Habilitar la API de Earth Engine en el proyecto
Con el proyecto creado y seleccionado, ir a https://console.cloud.google.com/apis/library/earthengine.googleapis.com y hacer clic en 'Habilitar'
5. Registrar el proyecto para uso con Earth Engine
Ir a https://console.cloud.google.com/earth-engine/configuration y seguir los indicando que es para investigación/uso académico no comercial.


In [ ]:
GEE_PROJECT = 'ee-seaceved'

import ee

print("Autenticando...")
ee.Authenticate()

print(f"Inicializando proyecto: {GEE_PROJECT}...")
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✓ Google Earth Engine inicializado correctamente!")

    info = ee.Number(1).getInfo()
    print("✓ Conexión verificada.")

except ee.EEException as e:
    print(f"\n✗ Falló la inicialización: {e}")
    if "not registered" in str(e):
        print(f"\nACCIÓN REQUERIDA: El proyecto '{GEE_PROJECT}' no está registrado en Earth Engine.")
        print(f"1. Regístralo aquí: https://console.cloud.google.com/earth-engine/configuration?project={GEE_PROJECT}")
        print("2. O actualiza la variable GEE_PROJECT con un Project ID válido.")
    elif "permission" in str(e).lower():
        print(f"\nACCIÓN REQUERIDA: Puede que no tengas permisos para el proyecto '{GEE_PROJECT}'.")
        print("Revisa la configuración de IAM en Google Cloud Console.")
except Exception as e:
    print(f"\n✗ Ocurrió un error inesperado: {e}")

Autenticando...
Inicializando proyecto: ee-seaceved...
✓ Google Earth Engine inicializado correctamente!
✓ Conexión verificada.


In [ ]:
# Leemos una cuenca del inventario de erosion de CIREN
# Yo elegi la del Estero Cholqui https://inventarioerosion.ciren.cl/layers/geonode_data:geonode:r13_168_rep_1ha
import geopandas as gpd

url = "https://inventarioerosion.ciren.cl/geoserver/ows?service=WFS&version=1.0.0&request=GetFeature&typename=geonode%3Ar13_168_rep_1ha&outputFormat=json&srs=EPSG%3A32719&srsName=EPSG%3A32719"
gp_df = gpd.read_file(url)
gp_df_simple = gp_df.dissolve() # opcion de geopandas para solo tener el contorno de la cuenca
area_de_interes = geemap.geopandas_to_ee(gp_df_simple) # de geopandas a gee


In [ ]:
# Visualizamos el area de interes
mapa_simple = geemap.Map()
mapa_simple.centerObject(area_de_interes, zoom=12)
mapa_simple.addLayer(area_de_interes, {'color': 'black'}, 'Área de interés')
mapa_simple

Map(center=[-33.809034207791086, -71.08282686452951], controls=(WidgetControl(options=['position', 'transparen…

Vamos a comenzar con el primer componente de RUSLE: erosividad de la lluvia.Según Bonilla y Vidal 2011, R = 0.0028P^1.534, donde P es la precipitación anual en mm. Vamos a cargar las precipitaciones de un año hidrológico (inicia 1 de abril y termina el 31 de marzo) solo como ejemplo para recordar el concepto. Pero por lo general los estudios son anuales (1 enero a 1 de diciembre)


In [ ]:
# 1. Cargar precipitación CHIRPS para el año hidrológico (1 abril - 31 marzo)
precipitacion = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterBounds(area_de_interes) \
    .filterDate('2023-04-01', '2024-03-31')

# 2. Sumar toda la precipitación del período y recortar al AOI
precip_anual_img = precipitacion.sum().clip(area_de_interes)

# 3. Calcular R como raster (variación espacial dentro del AOI)
R_img = precip_anual_img.pow(1.534).multiply(0.0028)

# 4. Visualizar en el mapa
viz_R = {'min': 10, 'max': 50, 'palette': ['white', 'yellow', 'orange', 'red']}

stats_R = R_img.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes,
    scale=250
).getInfo()

print("Rango de R (min/max):", stats_R)

mapa_simple.addLayer(R_img, viz_R, 'Erosividad R')
mapa_simple

Rango de R (min/max): {'precipitation_max': 31.27903562875093, 'precipitation_min': 23.133998711405418}


Map(bottom=157557.0, center=[-33.80767911381059, -70.63934326171876], controls=(WidgetControl(options=['positi…

Ahora vamos a descargar datos de suelo (arena, limo y arcilla + materia orgánica y usar la fórmula de EPIC para obtener valores de K. Los obtendremos de OpenLandMap https://stac.openlandmap.org/organic.carbon_usda.6a1c/collection.json?.language=en

In [ ]:

# Carbono orgánico del suelo (g/kg, capa 0 cm)
carbono_organico = ee.Image('OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arcilla (% en masa, capa 0 cm)
arcilla = ee.Image('OpenLandMap/SOL/SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Arena (% en masa, capa 0 cm)
arena = ee.Image('OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02') \
    .select('b0') \
    .clip(area_de_interes)

# Limo (% en masa, capa 0 cm)
limo = ee.Image(100).subtract(arena).subtract(arcilla)


In [ ]:
carbono_organico_pct = carbono_organico.multiply(0.5)  # OpenLandMap unidades en g/kg → %

sn1 = ee.Image(1).subtract(arena.divide(100)) # fraccion que no es arena

# f(csand) = 0.2 + 0.3*exp[-0.256*arena*(1 - silt/100)]
factor1 = ee.Image(0.2).add(
    ee.Image(0.3).multiply(
        arena.multiply(-0.256).multiply(
            ee.Image(1).subtract(limo.divide(100))
        ).exp()
    )
)

# f(cl-si) = (limo / (arcilla + limo))^0.3
factor2 = (limo.divide(arcilla.add(limo))).pow(0.3)

# f(orgc) = 1 - 0.25*orgC / (orgC + exp[3.72 - 2.95*orgC])
factor3 = ee.Image(1).subtract(
    ee.Image(0.25).multiply(carbono_organico_pct).divide(
        carbono_organico_pct.add(
            (ee.Image(3.72).subtract(carbono_organico_pct.multiply(2.95))).exp()
        )
    )
)

# f(hisand) = 1 - 0.7*SN1 / (SN1 + exp[-5.51 + 22.9*SN1])
factor4 = ee.Image(1).subtract(
    ee.Image(0.7).multiply(sn1).divide(
        sn1.add((ee.Image(-5.51).add(sn1.multiply(22.9))).exp())
    )
)

stats_K = K_img.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=area_de_interes,
    scale=250
).getInfo()

print("Rango de K (min/max):", stats_K)

K_img = factor1.multiply(factor2).multiply(factor3).multiply(factor4)

viz_K = {'min': 0.1, 'max': 0.25, 'palette': ['white', 'lightgreen', 'green', 'darkgreen']}
mapa_simple.addLayer(K_img, viz_K, 'Erodabilidad K')
mapa_simple

Rango de K (min/max): {'constant_max': 0.1411128934380734, 'constant_min': 0.11660228131826168}


Map(bottom=629254.0, center=[-33.78656650948914, -71.0973358154297], controls=(WidgetControl(options=['positio…